# Temporary Python 3.12 Browser Notebook

This notebook runs independently in each visitor's browser. Use **Load attachments**, then run the cells in order or select **Run all**.

| Feature | Example |
|---|---|
| Stateful execution | Variables are reused by later cells |
| Data files | CSV, JSON, SQL, and image attachments |
| Scientific stack | NumPy, pandas, SQLite, Matplotlib, Pillow |

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

UPLOADS = Path("uploads")
print(f"Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"Uploaded files: {[path.name for path in sorted(UPLOADS.glob('*'))]}")

In [ ]:
csv_path = UPLOADS / "temporary-demo-data.csv"
config_path = UPLOADS / "temporary-demo-config.json"

if not csv_path.exists():
    raise FileNotFoundError("Select Load attachments before running this cell.")

scores = pd.read_csv(csv_path)
config = json.loads(config_path.read_text(encoding="utf-8"))
summary = {
    "mean": float(np.mean(scores["score"])),
    "median": float(np.median(scores["score"])),
    "target": config["target_mean"],
}
display(scores)
summary

In [ ]:
query = (UPLOADS / "temporary-demo-query.sql").read_text(encoding="utf-8")
with sqlite3.connect(":memory:") as connection:
    connection.execute("CREATE TABLE scores (item TEXT, score REAL, group_name TEXT)")
    connection.executemany(
        "INSERT INTO scores VALUES (?, ?, ?)",
        scores[["item", "score", "group"]].itertuples(index=False, name=None),
    )
    sql_summary = pd.read_sql_query(query, connection)
sql_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.bar(scores["item"], scores["score"], color=config["theme_color"])
plt.axhline(summary["mean"], color="#cf222e", linestyle="--", label=f"Mean: {summary['mean']:.1f}")
plt.ylim(0, 100)
plt.ylabel("Score")
plt.title("Temporary Browser Notebook Demo")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

image_path = UPLOADS / "temporary-classroom-sketch.jpg"
with Image.open(image_path) as image:
    image_info = {"format": image.format, "size": image.size, "mode": image.mode}
image_info

In [ ]:
output_path = Path("temporary-summary.csv")
sql_summary.to_csv(output_path, index=False)
print(f"Created {output_path} ({output_path.stat().st_size} bytes)")